<a href="https://colab.research.google.com/github/C-yue28/FreeCodeCamp-ML-Projects/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip uninstall -y tensorflow tensorflow-intel
# !pip install tensorflow==2.20.0

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Embedding, Dense, TextVectorization
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
headers = ["class", "msg"]

train_data = pd.read_csv(train_file_path, sep='\t', names=headers)
test_data = pd.read_csv(test_file_path, sep='\t', names=headers)

train_dataset = train_data["msg"].values.tolist()
train_labels = np.array([1 if label=="spam" else 0 for label in train_data["class"].values.tolist()])
test_dataset = test_data["msg"].values.tolist()
test_labels = np.array([1 if label=="spam" else 0 for label in test_data["class"].values.tolist()])

# vocabulary = {}
# for msg in train_dataset:
#   for word in msg.split():
#     vocabulary[word] = vocabulary[word] + 1 if word in vocabulary else 1

# VOCAB_SIZE = len(vocabulary)
# MAX_LEN = len(max(train_dataset, key=lambda entry: len(entry.split())).split())

# print(VOCAB_SIZE)
# print(MAX_LEN)

vectorizer = TextVectorization(
    max_tokens=10000,
    output_sequence_length=50,
    standardize='lower_and_strip_punctuation'
)
vectorizer.adapt(train_dataset)
vectorized_train_dataset = vectorizer(train_dataset)
vectorized_test_dataset = vectorizer(test_dataset)

In [ ]:
model = Sequential([
  Embedding(input_dim=10000, output_dim=64, input_length=50),
  Flatten(),
  Dense(32, activation="relu"),
  Dense(1, activation="sigmoid")
])

monitor = EarlyStopping(monitor='val_acc', min_delta=0.00001, patience=20, verbose=1, mode='max', restore_best_weights=True)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
model.fit(vectorized_train_dataset, train_labels, validation_data=(vectorized_test_dataset, test_labels), epochs=100, callbacks=[monitor])

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):

  vectorized_text = vectorizer([pred_text])
  prediction = model.predict(vectorized_text)[0][0]
  prediction = [float(prediction), "ham" if np.round(prediction) == 0 else "spam"]

  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
